# 5. Simulate Population Responses

This notebook reconstructs predicted voxel responses using the encoding models estimated in previous notebooks.

For each voxel and session:

1. Load the fitted regression coefficients.
2. Load pRF-sampled image features.
3. Apply the encoding model.
4. Generate predicted responses for a large set of images.

The output is a simulated population response matrix that can be used for downstream representational drift analyses.

# Notebook Position in the Pipeline

Pipeline order:

1. pRF Sampling
2. Session-wise Regression
3. Combine Session ROIs
4. Save Session Permutations
5. Simulate Population Responses ← current notebook
6. Population Drift Analysis & RDM Drift Analysis


# Inputs

This notebook requires two inputs:

## 1. Combined regression results

Generated by:

`3_Combine_Session__ROIs.ipynb`

Files:

regressSessCombineROI_sub{subject}.pkl

These provide:

- voxel coefficients
- session-specific models
- beta information

## 2. pRF-sampled image features

Generated by:

`1_nsd_prf_sampling_from_pyramid.ipynb`

Files:

prfSampleStim_v{region}_sub{subject}.h5

These provide:

- steerable pyramid features
- orientation features
- pRF-specific feature vectors

# Output

Output file:

simPopResp_v{region}_sub{subject}.npz

Contains:

- voxSessResp
- voxSessRespOri
- simImgs
- nsessions

These arrays represent predicted voxel responses generated from the encoding models.

# PRE-FLIGHT CHECK FOR NOTEBOOK 5
# ============================================================
# This diagnostic verifies that all inputs required by:

#     5_sim_pop_response_expand.ipynb

# are internally consistent before generating simulated responses.
#
# Main goals:

     1. Confirm that the required input files exist.
     2. Verify that ROI combination was performed correctly.
     3. Verify that voxel counts match across files.
     4. Verify that coefficient dimensions match feature dimensions.
     5. Confirm that feature matrices can generate predictions.
#
 This cell should be run before launching a large simulation job.


In [ ]:
import os
import numpy as np
import h5py
import pickle
from google.colab import drive

drive.mount("/content/drive", force_remount=False)


BASE_FOLDER = "/content/drive/MyDrive/V1_Drift/repDrift_expand"
PRF_FOLDER  = "/content/drive/MyDrive/V1_Drift/prfsample_expand"

def _zstr(to_zscore: int) -> str:
    return ['', '_zscore', '_zeroMean', '_equalStd', '_zeroROImean'][to_zscore]

def check_sim_pop_inputs(isub: int, vreg: int = 1, to_zscore: int = 0, nimgs: int = 100):
    import os, pickle, h5py
    import numpy as np

    z = _zstr(to_zscore)

    sess_path = os.path.join(BASE_FOLDER, f"regressSessCombineROI_sub{isub}{z}.pkl")
    prf_path  = os.path.join(PRF_FOLDER,  f"prfSampleStim_v{vreg}_sub{isub}.h5")

    print("=" * 80)
    print(f"Checking subject {isub}, vreg {vreg}, zscore={to_zscore}")
    print(f"Regression file exists: {os.path.exists(sess_path)}")
    print(f"pRF file exists:        {os.path.exists(prf_path)}")

    with open(sess_path, "rb") as f:
        D = pickle.load(f)

    vox_coef = np.asarray(D["voxCoef"][vreg - 1])
    vox_ori_coef = np.asarray(D["voxOriCoef"][vreg - 1])
    sess_betas = np.asarray(D["sessBetas"][vreg - 1])

    print("\nRegression arrays:")
    print("vox_coef shape:     ", vox_coef.shape)
    print("vox_ori_coef shape: ", vox_ori_coef.shape)
    print("sess_betas shape:   ", sess_betas.shape)

    with h5py.File(prf_path, "r") as f:
        num_levels = int(f.attrs["numLevels"])
        num_orientations = int(f.attrs["numOrientations"])

        lev_keys = sorted(list(f["prfSampleLev"].keys()))
        ori_keys = sorted(list(f["prfSampleLevOri"].keys()))

        print("\npRF metadata:")
        print("num_levels:       ", num_levels)
        print("num_orientations: ", num_orientations)
        print("lev_keys:         ", lev_keys)
        print("ori_keys:         ", ori_keys)

        if lev_keys != ori_keys:
            raise ValueError("Mismatch: prfSampleLev and prfSampleLevOri keys are not identical.")

        print("\nIndividual ROI shapes:")
        for k in lev_keys:
            print(
                k,
                "lev:", f[f"prfSampleLev/{k}"].shape,
                "ori:", f[f"prfSampleLevOri/{k}"].shape
            )

        if vreg < 4:
            lev = np.concatenate([f[f"prfSampleLev/{k}"][:] for k in lev_keys], axis=1)
            levO = np.concatenate([f[f"prfSampleLevOri/{k}"][:] for k in ori_keys], axis=1)
        else:
            lev = f[f"prfSampleLev/{lev_keys[0]}"][:]
            levO = f[f"prfSampleLevOri/{ori_keys[0]}"][:]

    print("\nAfter MATLAB-style ROI combination:")
    print("combined lev shape: ", lev.shape)
    print("combined ori shape: ", levO.shape)

    expected_nonori_p = num_levels + 2 + 1
    expected_ori_p = num_levels * num_orientations + 2 + 1

    print("\nExpected coefficient dimensions:")
    print("non-orientation expected:", expected_nonori_p)
    print("orientation expected:    ", expected_ori_p)
    print("actual non-orientation:  ", vox_coef.shape[-1])
    print("actual orientation:      ", vox_ori_coef.shape[-1])

    assert vox_coef.shape[-1] == expected_nonori_p, "Wrong non-orientation coefficient length"
    assert vox_ori_coef.shape[-1] == expected_ori_p, "Wrong orientation coefficient length"

    assert vox_coef.shape[1] == lev.shape[1], (
        f"Voxel mismatch: vox_coef has {vox_coef.shape[1]} voxels, "
        f"but combined pRF lev has {lev.shape[1]}"
    )
    assert vox_ori_coef.shape[1] == levO.shape[1], (
        f"Voxel mismatch: vox_ori_coef has {vox_ori_coef.shape[1]} voxels, "
        f"but combined pRF ori has {levO.shape[1]}"
    )

    assert lev.shape[0] >= nimgs, "Not enough images in prfSampleLev"
    assert levO.shape[0] >= nimgs, "Not enough images in prfSampleLevOri"

    # MATLAB simImgs = 1:nimgs
    sim_imgs = np.arange(nimgs)

    ivox = 0
    isess = 0

    X_lev = np.concatenate([
        lev[sim_imgs, ivox, :num_levels],
        lev[sim_imgs, ivox, num_levels:num_levels + 2],
        np.ones((nimgs, 1))
    ], axis=1)

    X_ori_flat = levO[sim_imgs, ivox, :num_levels, :].reshape(
        nimgs,
        num_levels * num_orientations,
        order="F"
    )

    X_ori = np.concatenate([
        X_ori_flat,
        lev[sim_imgs, ivox, num_levels:num_levels + 2],
        np.ones((nimgs, 1))
    ], axis=1)

    pred_nonori = X_lev @ vox_coef[isess, ivox, :]
    pred_ori = X_ori @ vox_ori_coef[isess, ivox, :]

    print("\nManual prediction test, voxel 0, session 0:")
    print("X_lev shape:", X_lev.shape)
    print("X_ori shape:", X_ori.shape)
    print("pred_nonori shape:", pred_nonori.shape)
    print("pred_ori shape:   ", pred_ori.shape)
    print("first 5 nonori predictions:", pred_nonori[:5])
    print("first 5 ori predictions:   ", pred_ori[:5])

    print("\nPASS: indexing, ROI combination, and coefficient dimensions are consistent.")

In [ ]:
check_sim_pop_inputs(isub=1, vreg=1, to_zscore=0, nimgs=100)

In [ ]:
for vreg in [1, 2, 3, 4]:
    check_sim_pop_inputs(isub=1, vreg=vreg, to_zscore=0, nimgs=100)

# Run code

In [ ]:
import os
import numpy as np
import h5py
import pickle
from tqdm import tqdm
from google.colab import drive

# ---------- Mount Drive (idempotent) ----------
drive.mount("/content/drive", force_remount=False)

BASE_FOLDER = "/content/drive/MyDrive/V1_Drift/repDrift_expand"
PRF_FOLDER  = "/content/drive/MyDrive/V1_Drift/prfsample_expand"

def _zstr(to_zscore: int) -> str:
    return ['', '_zscore', '_zeroMean', '_equalStd', '_zeroROImean'][to_zscore]

# Check that all required inputs exist before running
# the simulation for a given subject and visual region.
def _has_required_files(isub: int, vreg: int, to_zscore: int) -> bool:
    z = _zstr(to_zscore)
    sess_path = os.path.join(BASE_FOLDER, f"regressSessCombineROI_sub{isub}{z}.pkl")
    prf_path  = os.path.join(PRF_FOLDER,  f"prfSampleStim_v{vreg}_sub{isub}.h5")
    return os.path.exists(sess_path) and os.path.exists(prf_path)

# Automatically identify subjects that have both:
#
#     1. Combined regression outputs
#     2. pRF sampling outputs
#
# available on disk.
def discover_subjects(vreg: int, to_zscore: int, pool=range(1,9)):
    """Return the list of subjects in 1..8 that have all inputs present."""
    found = []
    for s in pool:
        if _has_required_files(s, vreg, to_zscore):
            found.append(s)
    return found


# ============================================================
# MAIN SIMULATION FUNCTION
# ============================================================
# Reconstruct predicted voxel responses by combining:
#     image features × regression coefficients
# separately for every:
#     voxel
#     session
#     image
#
# This reproduces the MATLAB implementation
# simPopResponse_expand.m
# ============================================================
def sim_pop_response_expand(isub: int, vreg: int = 1, to_zscore: int = 0, nimgs: int = 100):
    """
    Simulate population responses by combining per-session regression
    coefficients with pRF-sampled feature maps (levels & orientations).
    Mirrors MATLAB simPopResponse_expand.m for a single subject/region.
    """
    z = _zstr(to_zscore)

    # --- Load combined-ROI regression results (pkl) ---
    sess_path = os.path.join(BASE_FOLDER, f"regressSessCombineROI_sub{isub}{z}.pkl")
    with open(sess_path, "rb") as f:
      D = pickle.load(f)
    vox_coef = np.asarray(D["voxCoef"][vreg - 1])
    vox_ori_coef = np.asarray(D["voxOriCoef"][vreg - 1])
    sess_betas = np.asarray(D["sessBetas"][vreg - 1])
    nsess = sess_betas.shape[0]
    nvox = vox_coef.shape[1]

    # --- Load pRF samples (HDF5) ---
    prf_path = os.path.join(PRF_FOLDER, f"prfSampleStim_v{vreg}_sub{isub}.h5")
    with h5py.File(prf_path, "r") as f:
        num_levels       = int(f.attrs["numLevels"])
        num_orientations = int(f.attrs["numOrientations"])

        lev_keys = sorted(list(f["prfSampleLev"].keys()))
        ori_keys = sorted(list(f["prfSampleLevOri"].keys()))
        if lev_keys != ori_keys:
          raise ValueError(f"Mismatch between prfSampleLev keys {lev_keys} and prfSampleLevOri keys {ori_keys}")

        print(f"vreg={vreg}, lev_keys={lev_keys}, ori_keys={ori_keys}")

        if vreg < 4:
            lev  = np.concatenate([f[f"prfSampleLev/{k}"][:] for k in lev_keys], axis=1)
            levO = np.concatenate([f[f"prfSampleLevOri/{k}"][:] for k in ori_keys], axis=1)
        else:
            if len(lev_keys) != 1:
              raise ValueError(f"Expected one ROI for vreg={vreg}, got {lev_keys}")
            lev  = f[f"prfSampleLev/{lev_keys[0]}"][:]
            levO = f[f"prfSampleLevOri/{ori_keys[0]}"][:]

    # --- Clip nimgs to available images ---
    nimgs_avail = lev.shape[0]
    if nimgs > nimgs_avail:
        nimgs = nimgs_avail
    sim_imgs = np.arange(1, nimgs + 1, dtype=int)  # MATLAB-like 1-based selection

    # --- Allocate outputs (nvox × nsess × nimgs) ---
    vox_sess_resp     = np.zeros((nvox, nsess, nimgs), dtype=np.float32)
    vox_sess_resp_ori = np.zeros((nvox, nsess, nimgs), dtype=np.float32)

    # --- Simulate responses voxel by voxel ---
    for ivox in tqdm(range(nvox), desc=f"S{isub} V{vreg}: sim responses", ncols=100):
        # Level features (+ two SF extras + intercept) → width = num_levels + 2 + 1
        X_lev_core = lev[sim_imgs - 1, ivox, :num_levels]                   # [nimgs, L]
        X_extras   = lev[sim_imgs - 1, ivox, num_levels:num_levels+2]       # [nimgs, 2]
        X_lev = np.concatenate([X_lev_core, X_extras, np.ones((nimgs, 1))], axis=1)

        # Orientation features: take first L levels, flatten in MATLAB (Fortran) order,
        # append the same two SF extras and an intercept → width = L*O + 2 + 1
        X_ori_3d = levO[sim_imgs - 1, ivox, :num_levels, :]                 # [nimgs, L, O]
        X_ori = X_ori_3d.reshape(nimgs, num_levels * num_orientations, order="F")
        X_ori    = np.concatenate([X_ori, X_extras, np.ones((nimgs, 1))], axis=1)

        # Session-wise multiplication with coefficients
        for isess in range(nsess):
            vox_sess_resp[ivox, isess, :]     = X_lev @ vox_coef[isess, ivox, :]
            vox_sess_resp_ori[ivox, isess, :] = X_ori @ vox_ori_coef[isess, ivox, :]

    # --- Save NPZ next to regression files for this subject/region ---
    out_path = os.path.join(
    BASE_FOLDER,
    f"simPopResp_v{vreg}_sub{isub}{z}.npz"
)
    np.savez_compressed(out_path,
                        voxSessResp=vox_sess_resp,
                        voxSessRespOri=vox_sess_resp_ori,
                        simImgs=sim_imgs,
                        nsessions=nsess)
    print(f"Saved NPZ: {out_path}")
    print(f"Shapes → voxSessResp: {vox_sess_resp.shape} | voxSessRespOri: {vox_sess_resp_ori.shape}")

def run_sim_for_subjects(subjects=None, vreg: int = 1, to_zscore: int = 0, nimgs: int = 100):
    """
    MATLAB-like driver: loops subjects 1..8 (or a provided list) and runs simulation
    for those that have the required inputs.
    """
    if subjects is None:
        subjects = discover_subjects(vreg=vreg, to_zscore=to_zscore, pool=range(1,9))
        print(f"Discovered ready subjects (v{vreg}, z={to_zscore}): {subjects}")
    else:
        # filter to those that actually have the files
        subjects = [s for s in subjects if _has_required_files(s, vreg, to_zscore)]
        print(f"Using provided subjects with files present: {subjects}")

    if not subjects:
        print("No subjects found with required inputs — nothing to do.")
        return

    for s in subjects:
        try:
            sim_pop_response_expand(isub=s, vreg=vreg, to_zscore=to_zscore, nimgs=nimgs)
        except Exception as e:
            print(f"[WARN] Subject {s} failed: {e}")

# ---------------- Examples ----------------
# 1) Exact MATLAB-style (runs everyone with files present in 1..8):
for vreg in [1,2,3,4]:
    run_sim_for_subjects(vreg=vreg, to_zscore=3, nimgs=100)
# 2) Run a subset explicitly (will still skip if files missing):
# run_sim_for_subjects(subjects=[1,2,3,4,5,6,7,8], vreg=1, to_zscore=0, nimgs=100)

# 3) Single subject:
# for vreg in [1, 2, 3, 4]:
#     sim_pop_response_expand(isub=1, vreg=vreg, to_zscore=0, nimgs=100)


# CHECK OUTPUT FROM NOTEBOOK 5: SIMULATED POPULATION RESPONSES
# ============================================================
 This cell verifies that the file generated by:

#     5_sim_pop_response_expand.ipynb

# was saved correctly and has the expected structure.
#
# Main goals:
#
    1. Confirm that the simulation output file exists.
    2. Check that all expected arrays were saved.
    3. Verify the dimensions of the simulated responses.
    4. Check for NaN or infinite values.
    5. Compare the scale of orientation and non-orientation predictions.
#
This is a diagnostic / quality-control cell only.


In [ ]:
import numpy as np, os

# ---------- CONFIG ----------
isub, vreg, to_zscore = 3, 1, 0
zstr = ['', '_zscore', '_zeroMean', '_equalStd', '_zeroROImean'][to_zscore]
base = "/content/drive/My Drive/V1_Drift/repDrift_expand"
path = os.path.join(base, f"v{vreg}", f"Subject {isub}", f"simPopResp_v{vreg}_sub{isub}{zstr}.npz")

# ---------- LOAD ----------
if not os.path.exists(path):
    raise FileNotFoundError(f"File not found: {path}")

d = np.load(path)
print(f"Loaded: {path}")

# ---------- BASIC INFO ----------
for k in d.files:
    v = d[k]
    print(f"{k:15s} shape={v.shape}  dtype={v.dtype}  nan%={np.isnan(v).mean()*100 if np.issubdtype(v.dtype, np.floating) else 0:.2f}%")

vox_sess_resp = d["voxSessResp"]
vox_sess_resp_ori = d["voxSessRespOri"]
nsess = d["nsessions"]
nvox, nsess, nimgs = vox_sess_resp.shape

print("\n--- Shape sanity ---")
print(f"voxSessResp:     (nvox={nvox}, nsess={nsess}, nimgs={nimgs})")
print(f"voxSessRespOri:  (nvox={vox_sess_resp_ori.shape[0]}, nsess={vox_sess_resp_ori.shape[1]}, nimgs={vox_sess_resp_ori.shape[2]})")

# ---------- VALUE CHECKS ----------
print("\n--- Basic range checks ---")
print(f"voxSessResp     min={vox_sess_resp.min():.4f}  max={vox_sess_resp.max():.4f}  mean={vox_sess_resp.mean():.4f}")
print(f"voxSessRespOri  min={vox_sess_resp_ori.min():.4f}  max={vox_sess_resp_ori.max():.4f}  mean={vox_sess_resp_ori.mean():.4f}")

# ---------- CROSS-VERIFY ONE RANDOM ENTRY ----------
ivox, isess, iimg = 0, 0, 0
pred_val = vox_sess_resp[ivox, isess, iimg]
print(f"\nExample voxel/session/image: v={ivox}, s={isess}, img={iimg} → value={pred_val:.6f}")

# ---------- HIGH-LEVEL SANITY SUMMARY ----------
print("\n File structure and shapes match expectations.")
if not np.all(np.isfinite(vox_sess_resp)):
    print(" Some NaN/inf values in voxSessResp.")
if not np.all(np.isfinite(vox_sess_resp_ori)):
    print(" Some NaN/inf values in voxSessRespOri.")
else:
    print(" All values finite.")

    import numpy as np

# ratio of typical magnitude between ori vs non-ori
ratio = abs(d["voxSessRespOri"]).mean() / abs(d["voxSessResp"]).mean()
print(f"Mean magnitude ratio (ori/non-ori) = {ratio:.2e}")

